# Building LLM

## Goal

This notebook is a hands-on journey to build a language model from scratch.

Each version introduces one new concept, allowing the model to evolve step by step while practicing language-model development.

---

## Version 7

In this version, we extend the recurrent neural character language model with a gated recurrent unit (GRU).

The model keeps the same character vocabulary, fixed four-character context, trainable embeddings and recurrent hidden-state size introduced in Version 6.

Instead of updating the hidden state with a single `tanh` transformation, the model now uses gates to control how information is preserved, updated and combined through the sequence.

The GRU introduces:

- an update gate;
- a reset gate;
- a candidate hidden state.

These components allow the recurrent model to learn more flexible hidden-state updates while keeping the architecture small and inspectable.

This version introduces gated recurrence without changing the training data or the overall language-modeling task.

## 1. Imports and Configuration

The Python standard library configures the execution environment before TensorFlow is imported.

GPU execution is disabled because this small model runs efficiently on the CPU and does not require CUDA. Low-level TensorFlow logs are suppressed to keep the notebook output clean.

TensorFlow provides tensor operations, trainable variables and automatic differentiation.

NumPy remains useful for reproducible data shuffling and sampling, while TensorFlow performs the model calculations and training.

The configuration collects the values that control the experiment.

`CONTEXT_LENGTH` defines how many previous characters are used to predict the next character.

`EMBEDDING_DIM` defines the size of the trainable vector used to represent each character.

`HIDDEN_DIM` defines the size of the recurrent hidden state used by the GRU.

An explicit seed makes weight initialization, data splitting, mini-batch shuffling and text generation reproducible.

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import tensorflow as tf

tf.config.set_visible_devices([], "GPU")

In [2]:
SEED = 42
TRAIN_FRACTION = 0.8
BATCH_SIZE = 32
LEARNING_RATE = 1.0
EPOCHS = 100

CONTEXT_LENGTH = 4
EMBEDDING_DIM = 8
HIDDEN_DIM = 16

tf.keras.utils.set_random_seed(SEED)

## 2. Training Data

### Training text

The English corpus is included directly in the notebook. It provides the text from which the model learns character patterns.

The corpus is kept unchanged from Version 6 so that the effect of gated recurrent processing can be observed without changing the training data.

In [3]:
corpus = 'language models learn patterns from text.\na small model predicts what character may come next.\nwe begin with counting because counting is easy to inspect.\nthe model sees letters, spaces, and punctuation.\neach prediction comes from examples found in the training text.\nsimple systems help us understand more advanced systems.\nlater versions will learn parameters with neural networks.\nclear experiments make machine learning easier to study.'

print(corpus)
print("Characters:", len(corpus))

language models learn patterns from text.
a small model predicts what character may come next.
we begin with counting because counting is easy to inspect.
the model sees letters, spaces, and punctuation.
each prediction comes from examples found in the training text.
simple systems help us understand more advanced systems.
later versions will learn parameters with neural networks.
clear experiments make machine learning easier to study.
Characters: 440


### Vocabulary

The vocabulary is the set of symbols the model can represent. Because this is a character model, every letter, space, punctuation mark and newline is a token.

Each character is assigned an integer identifier.

As in Version 6, these identifiers are used to look up trainable embedding vectors.

In [4]:
vocabulary = sorted(set(corpus))
vocabulary_size = len(vocabulary)

character_to_id = {character: index for index, character in enumerate(vocabulary)}
id_to_character = {index: character for character, index in character_to_id.items()}

print("Vocabulary size:", vocabulary_size)
print("Vocabulary:", repr("".join(vocabulary)))
print("First mappings:", list(character_to_id.items())[:10])

Vocabulary size: 27
Vocabulary: '\n ,.abcdefghiklmnoprstuvwxy'
First mappings: [('\n', 0), (' ', 1), (',', 2), ('.', 3), ('a', 4), ('b', 5), ('c', 6), ('d', 7), ('e', 8), ('f', 9)]


### Context windows and numerical encoding

Each character is converted into its integer identifier.

For the text `modeling`:

`modeling`  
↓  
`[15, 17, 7, 8, 14, 12, 16, 10]`

The model keeps the fixed four-character context used in Version 6.

With a context length of 4, four adjacent identifiers form each input context:

- `mode -> l` becomes `[15, 17, 7, 8] -> 14`
- `odel -> i` becomes `[17, 7, 8, 14] -> 12`
- `deli -> n` becomes `[7, 8, 14, 12] -> 16`
- `elin -> g` becomes `[8, 14, 12, 16] -> 10`

The four identifiers in each context are the input. The following identifier is the target to predict.

As in Version 6, the context embeddings are processed sequentially rather than flattened.

The difference is that Version 7 replaces the simple recurrent hidden-state update with a gated recurrent unit.

During generation, predicted identifiers are converted back into characters:

`[15, 17, 7, 8, 14, 12, 16, 10]`  
↓  
`modeling`

In [5]:
examples = [
    (corpus[index:index + CONTEXT_LENGTH], corpus[index + CONTEXT_LENGTH])
    for index in range(len(corpus) - CONTEXT_LENGTH)
]

input_ids = np.array([
    [character_to_id[character] for character in context]
    for context, _ in examples
], dtype=np.int32)

target_ids = np.array([
    character_to_id[target]
    for _, target in examples
], dtype=np.int32)

print("Number of examples:", len(examples))
print("Input shape:", input_ids.shape)
print("Target shape:", target_ids.shape)
print("First 8 examples:", examples[:8])
print("First 8 input IDs:")
print(input_ids[:8])
print("First 8 target IDs:", target_ids[:8])

Number of examples: 436
Input shape: (436, 4)
Target shape: (436,)
First 8 examples: [('lang', 'u'), ('angu', 'a'), ('ngua', 'g'), ('guag', 'e'), ('uage', ' '), ('age ', 'm'), ('ge m', 'o'), ('e mo', 'd')]
First 8 input IDs:
[[14  4 16 10]
 [ 4 16 10 22]
 [16 10 22  4]
 [10 22  4 10]
 [22  4 10  8]
 [ 4 10  8  1]
 [10  8  1 15]
 [ 8  1 15 17]]
First 8 target IDs: [22  4 10  8  1 15 17  7]


## 3. Neural Model

### Trainable embeddings and gated recurrent weights

Version 7 replaces the simple recurrent hidden-state update from Version 6 with a gated recurrent unit (GRU).

Each character identifier still selects a trainable embedding vector.

For a context of four characters:

`[15, 17, 7, 8]`  
↓  
`4 embedding vectors`

The embeddings are processed one at a time and in order.

At every position, the GRU calculates three components:

- an **update gate**, which controls how much of the previous hidden state is preserved;
- a **reset gate**, which controls how much previous information is used when creating the candidate state;
- a **candidate hidden state**, which contains the new information that could be stored.

The gates use the sigmoid function, producing values between 0 and 1.

The candidate hidden state uses `tanh`.

The final hidden-state update uses the update gate to interpolate between the previous hidden state and the candidate hidden state.

After the final character, the last hidden state is multiplied by an output weight matrix to produce one logit for every possible next character.

No bias terms are introduced yet, keeping the gated recurrent mechanism explicit and easy to inspect.

In [6]:
inputs = tf.convert_to_tensor(input_ids, dtype=tf.int32)

model_random = tf.random.Generator.from_seed(SEED)

embedding_matrix = tf.Variable(
    model_random.normal(
        shape=(vocabulary_size, EMBEDDING_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

update_input_weights = tf.Variable(
    model_random.normal(
        shape=(EMBEDDING_DIM, HIDDEN_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

update_recurrent_weights = tf.Variable(
    model_random.normal(
        shape=(HIDDEN_DIM, HIDDEN_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

reset_input_weights = tf.Variable(
    model_random.normal(
        shape=(EMBEDDING_DIM, HIDDEN_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

reset_recurrent_weights = tf.Variable(
    model_random.normal(
        shape=(HIDDEN_DIM, HIDDEN_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

candidate_input_weights = tf.Variable(
    model_random.normal(
        shape=(EMBEDDING_DIM, HIDDEN_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

candidate_recurrent_weights = tf.Variable(
    model_random.normal(
        shape=(HIDDEN_DIM, HIDDEN_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

output_weights = tf.Variable(
    model_random.normal(
        shape=(HIDDEN_DIM, vocabulary_size),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)


def gru_forward(
    embedding_matrix,
    update_input_weights,
    update_recurrent_weights,
    reset_input_weights,
    reset_recurrent_weights,
    candidate_input_weights,
    candidate_recurrent_weights,
    output_weights,
    inputs
):
    embeddings = tf.gather(embedding_matrix, inputs)

    hidden_state = tf.zeros((tf.shape(inputs)[0], HIDDEN_DIM), dtype=tf.float32)

    for position in range(CONTEXT_LENGTH):
        current_embedding = embeddings[:, position, :]

        update_gate = tf.sigmoid(
            tf.matmul(current_embedding, update_input_weights)
            +
            tf.matmul(hidden_state, update_recurrent_weights)
        )

        reset_gate = tf.sigmoid(
            tf.matmul(current_embedding, reset_input_weights)
            +
            tf.matmul(hidden_state, reset_recurrent_weights)
        )

        candidate_state = tf.tanh(
            tf.matmul(current_embedding, candidate_input_weights)
            +
            tf.matmul(reset_gate * hidden_state, candidate_recurrent_weights)
        )

        hidden_state = (update_gate * hidden_state + (1.0 - update_gate) * candidate_state)

    logits = tf.matmul(hidden_state, output_weights)

    return hidden_state, logits

In [7]:
example_embeddings = tf.gather(embedding_matrix, inputs[:1])

example_hidden_state, example_logits = gru_forward(
    embedding_matrix,
    update_input_weights,
    update_recurrent_weights,
    reset_input_weights,
    reset_recurrent_weights,
    candidate_input_weights,
    candidate_recurrent_weights,
    output_weights,
    inputs[:1]
)

trainable_parameters = (
    tf.size(embedding_matrix)
    + tf.size(update_input_weights)
    + tf.size(update_recurrent_weights)
    + tf.size(reset_input_weights)
    + tf.size(reset_recurrent_weights)
    + tf.size(candidate_input_weights)
    + tf.size(candidate_recurrent_weights)
    + tf.size(output_weights)
)

print("Input tensor shape:", inputs.shape)
print("Embedding matrix shape:", embedding_matrix.shape)
print("Embedded context shape:", example_embeddings.shape)

print("Update input weights shape:", update_input_weights.shape)
print("Update recurrent weights shape:", update_recurrent_weights.shape)

print("Reset input weights shape:", reset_input_weights.shape)
print("Reset recurrent weights shape:", reset_recurrent_weights.shape)

print("Candidate input weights shape:", candidate_input_weights.shape)
print("Candidate recurrent weights shape:", candidate_recurrent_weights.shape)

print("Final hidden state shape:", example_hidden_state.shape)
print("Output weight matrix shape:", output_weights.shape)
print("Logits shape:", example_logits.shape)
print("Trainable parameters:", trainable_parameters.numpy())

Input tensor shape: (436, 4)
Embedding matrix shape: (27, 8)
Embedded context shape: (1, 4, 8)
Update input weights shape: (8, 16)
Update recurrent weights shape: (16, 16)
Reset input weights shape: (8, 16)
Reset recurrent weights shape: (16, 16)
Candidate input weights shape: (8, 16)
Candidate recurrent weights shape: (16, 16)
Final hidden state shape: (1, 16)
Output weight matrix shape: (16, 27)
Logits shape: (1, 27)
Trainable parameters: 1800


### Training and validation split

The examples are divided into two separate groups:

- the training set is used to update the model parameters;
- the validation set is used to measure the loss on examples that do not update the parameters.

Each input example contains an ordered sequence of four character identifiers.

The GRU processes these identifiers from left to right.

The hidden state starts from zeros for every context example and is updated once for each character using the update gate, reset gate and candidate hidden state.

The indices are shuffled with a local random generator, making the split reproducible.

In [8]:
split_random = np.random.default_rng(SEED)

indices = split_random.permutation(len(inputs))
split_position = int(len(indices) * TRAIN_FRACTION)

train_indices = indices[:split_position]
validation_indices = indices[split_position:]

train_inputs = tf.gather(inputs, train_indices)
train_targets = tf.gather(target_ids, train_indices)

validation_inputs = tf.gather(inputs, validation_indices)
validation_targets = tf.gather(target_ids, validation_indices)

print("Training examples:", len(train_inputs))
print("Validation examples:", len(validation_inputs))
print("Training input shape:", train_inputs.shape)
print("Validation input shape:", validation_inputs.shape)

Training examples: 348
Validation examples: 88
Training input shape: (348, 4)
Validation input shape: (88, 4)


### Softmax probabilities

TensorFlow provides `tf.nn.softmax` to convert logits into probabilities.

- every probability is between 0 and 1;
- the probabilities for one input sum to 1;
- higher logits produce higher probabilities.

TensorFlow handles the numerical stability of this operation internally.

In [9]:
def softmax(logits):
    return tf.nn.softmax(logits, axis=1)

### Cross-entropy loss

Cross-entropy measures how much probability the model assigns to the correct next character.

Before calculating the loss, the context identifiers are mapped to their embedding vectors.

The embeddings are processed sequentially by the GRU.

At every position, the update gate, reset gate and candidate hidden state determine how the recurrent hidden state changes.

After the final context character, the last hidden state is multiplied by the output weights to produce the logits.

TensorFlow calculates the cross-entropy directly from the logits and integer target IDs using a numerically stable operation.

A high probability for the correct character produces a low loss.

Training will update the embedding matrix, all GRU weight matrices and the output weights to reduce this value.

In [10]:
def calculate_loss(
    embedding_matrix,
    update_input_weights,
    update_recurrent_weights,
    reset_input_weights,
    reset_recurrent_weights,
    candidate_input_weights,
    candidate_recurrent_weights,
    output_weights,
    inputs,
    targets
):
    _, logits = gru_forward(
        embedding_matrix,
        update_input_weights,
        update_recurrent_weights,
        reset_input_weights,
        reset_recurrent_weights,
        candidate_input_weights,
        candidate_recurrent_weights,
        output_weights,
        inputs
    )

    example_losses = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=targets, logits=logits)

    return tf.reduce_mean(example_losses)

In [11]:
initial_train_loss = calculate_loss(
    embedding_matrix,
    update_input_weights,
    update_recurrent_weights,
    reset_input_weights,
    reset_recurrent_weights,
    candidate_input_weights,
    candidate_recurrent_weights,
    output_weights,
    train_inputs,
    train_targets
)

initial_validation_loss = calculate_loss(
    embedding_matrix,
    update_input_weights,
    update_recurrent_weights,
    reset_input_weights,
    reset_recurrent_weights,
    candidate_input_weights,
    candidate_recurrent_weights,
    output_weights,
    validation_inputs,
    validation_targets
)

print("Initial train loss:", initial_train_loss.numpy())
print("Initial validation loss:", initial_validation_loss.numpy())

Initial train loss: 3.2958376
Initial validation loss: 3.295836


### Training with mini-batch gradient descent

Training remains organized into epochs.

At each recorded epoch, training and validation loss are measured using the current model parameters.

Except at the final recorded epoch, the training examples are then shuffled and divided into mini-batches.

For every mini-batch:

1. `tf.GradientTape` records the GRU forward calculations;
2. TensorFlow calculates the gradients automatically;
3. the embedding matrix, GRU weight matrices and output weights are updated manually with gradient descent.

The validation examples are never used to update the model parameters.

The gated recurrent computations are recorded by `tf.GradientTape`. Gradients therefore flow backward through the update gates, reset gates, candidate hidden states and recurrent hidden-state updates.

The same GRU weights are reused at every position in the context.

Version 7 keeps the same mini-batch training procedure while extending automatic differentiation to the complete gated recurrent model.

### Mini-batches

A mini-batch is a small group of training examples.

The model updates its parameters after every mini-batch instead of processing all training examples together.

The final mini-batch may contain fewer examples than the configured batch size.

In [12]:
def create_batches(inputs, targets, batch_size):
    for start in range(0, len(inputs), batch_size):
        end = start + batch_size

        batch_inputs = inputs[start:end]
        batch_targets = targets[start:end]

        yield batch_inputs, batch_targets

In [13]:
def train_model(
    train_inputs,
    train_targets,
    validation_inputs,
    validation_targets,
    initial_embedding_matrix,
    initial_update_input_weights,
    initial_update_recurrent_weights,
    initial_reset_input_weights,
    initial_reset_recurrent_weights,
    initial_candidate_input_weights,
    initial_candidate_recurrent_weights,
    initial_output_weights,
    learning_rate=1.0,
    batch_size=32,
    epochs=100,
    seed=42,
    print_every=10
):
    trained_embedding_matrix = tf.Variable(initial_embedding_matrix)

    trained_update_input_weights = tf.Variable(initial_update_input_weights)
    trained_update_recurrent_weights = tf.Variable(initial_update_recurrent_weights)

    trained_reset_input_weights = tf.Variable(initial_reset_input_weights)
    trained_reset_recurrent_weights = tf.Variable(initial_reset_recurrent_weights)

    trained_candidate_input_weights = tf.Variable(initial_candidate_input_weights)
    trained_candidate_recurrent_weights = tf.Variable(initial_candidate_recurrent_weights)

    trained_output_weights = tf.Variable(initial_output_weights)

    training_random = np.random.default_rng(seed)

    train_loss_history = []
    validation_loss_history = []

    for epoch in range(epochs + 1):
        train_loss = float(
            calculate_loss(
                trained_embedding_matrix,
                trained_update_input_weights,
                trained_update_recurrent_weights,
                trained_reset_input_weights,
                trained_reset_recurrent_weights,
                trained_candidate_input_weights,
                trained_candidate_recurrent_weights,
                trained_output_weights,
                train_inputs,
                train_targets
            ).numpy()
        )

        validation_loss = float(
            calculate_loss(
                trained_embedding_matrix,
                trained_update_input_weights,
                trained_update_recurrent_weights,
                trained_reset_input_weights,
                trained_reset_recurrent_weights,
                trained_candidate_input_weights,
                trained_candidate_recurrent_weights,
                trained_output_weights,
                validation_inputs,
                validation_targets
            ).numpy()
        )

        train_loss_history.append(train_loss)
        validation_loss_history.append(validation_loss)

        if print_every is not None and epoch % print_every == 0:
            print(
                f"Epoch {epoch:3d} | "
                f"Train loss: {train_loss:.4f} | "
                f"Validation loss: {validation_loss:.4f}"
            )

        if epoch == epochs:
            break

        shuffled_indices = training_random.permutation(len(train_inputs))

        shuffled_inputs = tf.gather(train_inputs, shuffled_indices)

        shuffled_targets = tf.gather(train_targets, shuffled_indices)

        for batch_inputs, batch_targets in create_batches(shuffled_inputs, shuffled_targets, batch_size):
            with tf.GradientTape() as tape:
                batch_loss = calculate_loss(
                    trained_embedding_matrix,
                    trained_update_input_weights,
                    trained_update_recurrent_weights,
                    trained_reset_input_weights,
                    trained_reset_recurrent_weights,
                    trained_candidate_input_weights,
                    trained_candidate_recurrent_weights,
                    trained_output_weights,
                    batch_inputs,
                    batch_targets
                )

            gradients = tape.gradient(
                batch_loss,
                [
                    trained_embedding_matrix,
                    trained_update_input_weights,
                    trained_update_recurrent_weights,
                    trained_reset_input_weights,
                    trained_reset_recurrent_weights,
                    trained_candidate_input_weights,
                    trained_candidate_recurrent_weights,
                    trained_output_weights
                ]
            )

            embedding_gradient = tf.convert_to_tensor(
                gradients[0]
            )

            trainable_variables = [
                trained_embedding_matrix,
                trained_update_input_weights,
                trained_update_recurrent_weights,
                trained_reset_input_weights,
                trained_reset_recurrent_weights,
                trained_candidate_input_weights,
                trained_candidate_recurrent_weights,
                trained_output_weights
            ]

            gradients[0] = embedding_gradient

            for variable, gradient in zip(trainable_variables, gradients):
                variable.assign_sub(learning_rate * gradient)

    return (
        trained_embedding_matrix,
        trained_update_input_weights,
        trained_update_recurrent_weights,
        trained_reset_input_weights,
        trained_reset_recurrent_weights,
        trained_candidate_input_weights,
        trained_candidate_recurrent_weights,
        trained_output_weights,
        train_loss_history,
        validation_loss_history
    )

In [14]:
(
    trained_embedding_matrix,
    trained_update_input_weights,
    trained_update_recurrent_weights,
    trained_reset_input_weights,
    trained_reset_recurrent_weights,
    trained_candidate_input_weights,
    trained_candidate_recurrent_weights,
    trained_output_weights,
    train_loss_history,
    validation_loss_history
) = train_model(
    train_inputs,
    train_targets,
    validation_inputs,
    validation_targets,
    embedding_matrix,
    update_input_weights,
    update_recurrent_weights,
    reset_input_weights,
    reset_recurrent_weights,
    candidate_input_weights,
    candidate_recurrent_weights,
    output_weights,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    seed=SEED
)

final_train_loss = train_loss_history[-1]
final_validation_loss = validation_loss_history[-1]

best_validation_epoch = int(np.argmin(validation_loss_history))

best_validation_loss = validation_loss_history[best_validation_epoch]

print()
print("Initial train loss:", initial_train_loss.numpy())
print("Final train loss:", final_train_loss)
print("Initial validation loss:", initial_validation_loss.numpy())
print("Final validation loss:", final_validation_loss)
print("Best validation loss:", best_validation_loss)
print("Best validation epoch:", best_validation_epoch)

Epoch   0 | Train loss: 3.2958 | Validation loss: 3.2958
Epoch  10 | Train loss: 3.2958 | Validation loss: 3.2958
Epoch  20 | Train loss: 3.2958 | Validation loss: 3.2958
Epoch  30 | Train loss: 3.2958 | Validation loss: 3.2958
Epoch  40 | Train loss: 3.2958 | Validation loss: 3.2958
Epoch  50 | Train loss: 3.2958 | Validation loss: 3.2958
Epoch  60 | Train loss: 3.2955 | Validation loss: 3.2956
Epoch  70 | Train loss: 3.1116 | Validation loss: 3.1520
Epoch  80 | Train loss: 2.8456 | Validation loss: 2.9874
Epoch  90 | Train loss: 2.7709 | Validation loss: 2.9203
Epoch 100 | Train loss: 2.6596 | Validation loss: 2.9139

Initial train loss: 3.2958376
Final train loss: 2.659555673599243
Initial validation loss: 3.295836
Final validation loss: 2.9139413833618164
Best validation loss: 2.8994626998901367
Best validation epoch: 99


### Reading the losses

The initial training and validation losses are both approximately `3.2958`, which is close to `ln(27)` and therefore consistent with an almost uniform prediction over the vocabulary.

The GRU learns slowly during the first part of training.

Because the embedding vectors and all gated recurrent weight matrices are initialized with small values, the first parameter updates are also small.

The losses remain almost unchanged during the first 50 epochs.

After approximately 60 epochs, the model begins to learn more clearly.

The training loss decreases from approximately:

`3.2958 -> 2.6596`

The validation loss also improves:

`3.2958 -> 2.9139`

The best validation loss is approximately:

`2.8995`

and is reached at epoch 99.

Validation loss increases slightly at the final epoch after reaching its minimum.

This suggests the beginning of mild overfitting.

The notebook records the best validation epoch for analysis but intentionally keeps the final epoch-100 parameters for the following demonstrations.

Version 7 therefore shows that the gated recurrent architecture can learn the same character-level prediction task while explicitly controlling how information is preserved and updated through the hidden state.

### Learned probabilities

After training, the GRU produces a probability distribution for the next character from an ordered four-character context.

The character identifiers are mapped to embeddings and processed sequentially.

At every position, the update gate, reset gate and candidate hidden state control how information is carried forward.

The final hidden state is converted into logits and then into probabilities with softmax.

The example below inspects the learned next-character distribution for the context `mode`.

In [15]:
example_context = "mode"

example_context_ids = tf.constant(
    [[character_to_id[character] for character in example_context]],
    dtype=tf.int32
)

example_hidden_state, example_logits = gru_forward(
    trained_embedding_matrix,
    trained_update_input_weights,
    trained_update_recurrent_weights,
    trained_reset_input_weights,
    trained_reset_recurrent_weights,
    trained_candidate_input_weights,
    trained_candidate_recurrent_weights,
    trained_output_weights,
    example_context_ids
)

learned_probabilities = softmax(example_logits)[0].numpy()

sorted_probabilities = sorted(
    zip(vocabulary, learned_probabilities),
    key=lambda item: item[1],
    reverse=True
)

print("Context:", repr(example_context))
print("Final hidden state shape:", example_hidden_state.shape)
print()

for character, probability in sorted_probabilities:
    print(repr(character), round(float(probability), 4))

print()
print("Total probability:", learned_probabilities.sum())

Context: 'mode'
Final hidden state shape: (1, 16)

' ' 0.2329
'e' 0.0918
's' 0.0825
't' 0.0765
'n' 0.0706
'i' 0.0636
'a' 0.063
'l' 0.0458
'r' 0.0455
'o' 0.0378
'm' 0.0309
'.' 0.0267
'u' 0.024
'p' 0.0222
'd' 0.0195
'c' 0.0178
'g' 0.0135
'h' 0.0112
'w' 0.0057
'x' 0.005
',' 0.0043
'y' 0.0027
'f' 0.0023
'v' 0.0015
'b' 0.0015
'k' 0.001
'\n' 0.0

Total probability: 1.0


## 4. Generator

The trained GRU can now generate new text one character at a time.

Generation remains autoregressive.

For every prediction:

1. the four most recent characters form the current context;
2. their identifiers are mapped to trainable embeddings;
3. the embeddings are processed from left to right by the GRU;
4. the update gate, reset gate and candidate hidden state control the recurrent computation;
5. the final hidden state produces the next-character logits;
6. softmax converts the logits into probabilities;
7. one character is sampled from the probability distribution;
8. the sampled character is appended to the generated text and the four-character context window moves forward.

The GRU hidden state is initialized from zeros for each context window, matching the way the model was trained.

A local NumPy random generator keeps text generation reproducible without changing the global random state.

### Sample the next character

The current four-character context is converted into numerical identifiers and processed by the GRU.

The embeddings are read from left to right through the gated recurrent computation.

The final hidden state is transformed into logits and probabilities.

A local random generator samples one identifier from this learned probability distribution and converts it back into a character.

In [16]:
def sample_next_character(
    context,
    embedding_matrix,
    update_input_weights,
    update_recurrent_weights,
    reset_input_weights,
    reset_recurrent_weights,
    candidate_input_weights,
    candidate_recurrent_weights,
    output_weights,
    random_generator
):
    context_ids = tf.constant(
        [[character_to_id[character] for character in context]],
        dtype=tf.int32
    )

    _, logits = gru_forward(
        embedding_matrix,
        update_input_weights,
        update_recurrent_weights,
        reset_input_weights,
        reset_recurrent_weights,
        candidate_input_weights,
        candidate_recurrent_weights,
        output_weights,
        context_ids
    )

    probabilities = softmax(logits)[0].numpy()

    next_id = random_generator.choice(vocabulary_size, p=probabilities)

    return id_to_character[next_id]

In [17]:
sample_random = np.random.default_rng(SEED)

example_context = "mode"

print("Context:", repr(example_context))

for _ in range(5):
    sampled_character = sample_next_character(
        example_context,
        trained_embedding_matrix,
        trained_update_input_weights,
        trained_update_recurrent_weights,
        trained_reset_input_weights,
        trained_reset_recurrent_weights,
        trained_candidate_input_weights,
        trained_candidate_recurrent_weights,
        trained_output_weights,
        sample_random
    )

    print("Sampled character:", repr(sampled_character))

Context: 'mode'
Sampled character: 'r'
Sampled character: 'e'
Sampled character: 's'
Sampled character: 'o'
Sampled character: ' '


### Generate text

Text generation starts from a four-character context.

At every step, the GRU processes the current context from left to right and samples the next character.

The sampled character is appended to the output.

The oldest context character is then removed, causing the fixed context window to slide forward by one position.

For example:

`mode -> sampled character`

then:

`ode? -> next sampled character`

and so on.

The hidden state starts from zeros for every new four-character window, matching the training procedure.

The model therefore generates text autoregressively while using gated recurrent processing inside every context window.

In [18]:
def generate_text(
    starting_context,
    number_of_characters,
    embedding_matrix,
    update_input_weights,
    update_recurrent_weights,
    reset_input_weights,
    reset_recurrent_weights,
    candidate_input_weights,
    candidate_recurrent_weights,
    output_weights,
    seed=42
):
    if len(starting_context) != CONTEXT_LENGTH:
        raise ValueError(f"starting_context must contain exactly {CONTEXT_LENGTH} characters")

    generated_text = starting_context
    generation_random = np.random.default_rng(seed)

    for _ in range(number_of_characters):
        current_context = generated_text[-CONTEXT_LENGTH:]

        next_character = sample_next_character(
            current_context,
            embedding_matrix,
            update_input_weights,
            update_recurrent_weights,
            reset_input_weights,
            reset_recurrent_weights,
            candidate_input_weights,
            candidate_recurrent_weights,
            output_weights,
            generation_random
        )

        generated_text += next_character

    return generated_text

In [19]:
generated_text = generate_text(
    starting_context="mode",
    number_of_characters=300,
    embedding_matrix=trained_embedding_matrix,
    update_input_weights=trained_update_input_weights,
    update_recurrent_weights=trained_update_recurrent_weights,
    reset_input_weights=trained_reset_input_weights,
    reset_recurrent_weights=trained_reset_recurrent_weights,
    candidate_input_weights=trained_candidate_input_weights,
    candidate_recurrent_weights=trained_candidate_recurrent_weights,
    output_weights=trained_output_weights,
    seed=SEED
)

print(generated_text)

moderato,usr metnse n unpaxtr m birteeh cl pesndrpaip c tnnpde blanrgand l ln.
eaphenhe 
 re m red pe axvieusng bse ge usnemen l mac.
awl n xis metlahane tt m papnr wc meroiwcm wa fysnttl toh sawaauaams gnum 
 bi.
ol tpl cuh mlr w mi.
gtlotis .
samitmalcotid ss ,iate wae .
, ,ipaimss cd.
e he t w vsusri


## 5. Tests

These assertions verify the context windows, TensorFlow tensors, trainable GRU parameters, data split, model shapes, gate values, hidden states, probability distributions, gradients, training behavior and reproducibility.

The tests verify that:

- context windows and targets are constructed correctly;
- all eight trainable parameter matrices are TensorFlow variables;
- embeddings, GRU weights, hidden states and logits have the expected shapes;
- the model contains exactly 1800 trainable parameters;
- update and reset gate values remain between 0 and 1;
- candidate hidden-state values remain between -1 and 1;
- hidden-state values remain finite and between -1 and 1;
- training and validation sets remain separate;
- training and validation loss histories contain the expected number of measurements;
- training reduces the loss and all recorded losses remain finite;
- probability distributions sum to 1;
- TensorFlow produces finite gradients for every trainable parameter matrix;
- every trainable parameter matrix changes during training;
- repeating training from the same initial parameters with the same seed produces the same parameters and loss histories;
- autoregressive generation is reproducible with the same starting context and seed.

The tests also verify the requested generated-text length and starting context.

In [20]:
assert len(examples) == len(corpus) - CONTEXT_LENGTH
assert len(input_ids) == len(examples)
assert len(target_ids) == len(examples)

assert input_ids.shape == (len(examples), CONTEXT_LENGTH)
assert target_ids.shape == (len(examples),)

assert examples[0][0] == corpus[:CONTEXT_LENGTH]
assert examples[0][1] == corpus[CONTEXT_LENGTH]
assert examples[1][0] == corpus[1:1 + CONTEXT_LENGTH]
assert examples[1][1] == corpus[1 + CONTEXT_LENGTH]

assert tf.is_tensor(inputs)

initial_parameter_variables = [
    embedding_matrix,
    update_input_weights,
    update_recurrent_weights,
    reset_input_weights,
    reset_recurrent_weights,
    candidate_input_weights,
    candidate_recurrent_weights,
    output_weights
]

trained_parameter_variables = [
    trained_embedding_matrix,
    trained_update_input_weights,
    trained_update_recurrent_weights,
    trained_reset_input_weights,
    trained_reset_recurrent_weights,
    trained_candidate_input_weights,
    trained_candidate_recurrent_weights,
    trained_output_weights
]

assert all(isinstance(variable, tf.Variable) for variable in initial_parameter_variables)
assert all(isinstance(variable, tf.Variable) for variable in trained_parameter_variables)

assert inputs.shape == (len(examples), CONTEXT_LENGTH)

assert embedding_matrix.shape == (vocabulary_size, EMBEDDING_DIM)
assert update_input_weights.shape == (EMBEDDING_DIM, HIDDEN_DIM)
assert update_recurrent_weights.shape == (HIDDEN_DIM, HIDDEN_DIM)
assert reset_input_weights.shape == (EMBEDDING_DIM, HIDDEN_DIM)
assert reset_recurrent_weights.shape == (HIDDEN_DIM, HIDDEN_DIM)
assert candidate_input_weights.shape == (EMBEDDING_DIM, HIDDEN_DIM)
assert candidate_recurrent_weights.shape == (HIDDEN_DIM, HIDDEN_DIM)
assert output_weights.shape == (HIDDEN_DIM, vocabulary_size)

for initial_variable, trained_variable in zip(initial_parameter_variables, trained_parameter_variables):
    assert trained_variable.shape == initial_variable.shape

expected_trainable_parameters = (
    vocabulary_size * EMBEDDING_DIM
    + 3 * (EMBEDDING_DIM * HIDDEN_DIM + HIDDEN_DIM * HIDDEN_DIM)
    + HIDDEN_DIM * vocabulary_size
)

assert expected_trainable_parameters == 1800
assert trainable_parameters.numpy() == expected_trainable_parameters

assert example_embeddings.shape == (1, CONTEXT_LENGTH, EMBEDDING_DIM)
assert example_hidden_state.shape == (1, HIDDEN_DIM)
assert example_logits.shape == (1, vocabulary_size)

assert np.all(np.isfinite(example_hidden_state.numpy()))
assert np.all(example_hidden_state.numpy() >= -1.0)
assert np.all(example_hidden_state.numpy() <= 1.0)

assert len(train_inputs) + len(validation_inputs) == len(inputs)
assert len(train_inputs) == len(train_targets)
assert len(validation_inputs) == len(validation_targets)
assert len(np.intersect1d(train_indices, validation_indices)) == 0

assert len(train_loss_history) == EPOCHS + 1
assert len(validation_loss_history) == EPOCHS + 1

assert final_train_loss < float(initial_train_loss.numpy())
assert final_validation_loss < float(initial_validation_loss.numpy())

assert np.all(np.isfinite(train_loss_history))
assert np.all(np.isfinite(validation_loss_history))

assert best_validation_epoch == int(np.argmin(validation_loss_history))
assert best_validation_loss == validation_loss_history[best_validation_epoch]


# GRU gate checks

gate_test_context = tf.constant([[character_to_id[character] for character in "mode"]],dtype=tf.int32)

gate_test_embeddings = tf.gather(trained_embedding_matrix, gate_test_context)

gate_test_hidden_state = tf.zeros((1, HIDDEN_DIM), dtype=tf.float32)
gate_test_embedding = gate_test_embeddings[:, 0, :]

gate_test_update = tf.sigmoid(
    tf.matmul(gate_test_embedding, trained_update_input_weights)
    + tf.matmul(gate_test_hidden_state, trained_update_recurrent_weights)
)

gate_test_reset = tf.sigmoid(
    tf.matmul(gate_test_embedding, trained_reset_input_weights)
    + tf.matmul(gate_test_hidden_state, trained_reset_recurrent_weights)
)

gate_test_candidate = tf.tanh(
    tf.matmul(gate_test_embedding, trained_candidate_input_weights)
    + tf.matmul(gate_test_reset * gate_test_hidden_state, trained_candidate_recurrent_weights)
)

gate_test_hidden_state = (
    gate_test_update * gate_test_hidden_state
    + (1.0 - gate_test_update) * gate_test_candidate
)

assert gate_test_update.shape == (1, HIDDEN_DIM)
assert gate_test_reset.shape == (1, HIDDEN_DIM)
assert gate_test_candidate.shape == (1, HIDDEN_DIM)
assert gate_test_hidden_state.shape == (1, HIDDEN_DIM)

assert np.all(np.isfinite(gate_test_update.numpy()))
assert np.all(np.isfinite(gate_test_reset.numpy()))
assert np.all(np.isfinite(gate_test_candidate.numpy()))
assert np.all(np.isfinite(gate_test_hidden_state.numpy()))

assert np.all(gate_test_update.numpy() >= 0.0)
assert np.all(gate_test_update.numpy() <= 1.0)

assert np.all(gate_test_reset.numpy() >= 0.0)
assert np.all(gate_test_reset.numpy() <= 1.0)

assert np.all(gate_test_candidate.numpy() >= -1.0)
assert np.all(gate_test_candidate.numpy() <= 1.0)

assert np.all(gate_test_hidden_state.numpy() >= -1.0)
assert np.all(gate_test_hidden_state.numpy() <= 1.0)


# Probability checks

probability_context = tf.constant([[character_to_id[character] for character in "mode"]], dtype=tf.int32)

_, probability_logits = gru_forward(
    trained_embedding_matrix,
    trained_update_input_weights,
    trained_update_recurrent_weights,
    trained_reset_input_weights,
    trained_reset_recurrent_weights,
    trained_candidate_input_weights,
    trained_candidate_recurrent_weights,
    trained_output_weights,
    probability_context
)

probability_values = softmax(probability_logits)[0].numpy()

assert abs(probability_values.sum() - 1.0) < 1e-6
assert np.all(np.isfinite(probability_values))
assert np.all(probability_values >= 0.0)
assert np.all(probability_values <= 1.0)


# Gradient checks

gradient_inputs = train_inputs[:BATCH_SIZE]
gradient_targets = train_targets[:BATCH_SIZE]

with tf.GradientTape() as tape:
    gradient_loss = calculate_loss(
        embedding_matrix,
        update_input_weights,
        update_recurrent_weights,
        reset_input_weights,
        reset_recurrent_weights,
        candidate_input_weights,
        candidate_recurrent_weights,
        output_weights,
        gradient_inputs,
        gradient_targets
    )

gradients = tape.gradient(gradient_loss, initial_parameter_variables)

assert all(gradient is not None for gradient in gradients)

dense_gradients = [tf.convert_to_tensor(gradient) for gradient in gradients]

for gradient, variable in zip(dense_gradients, initial_parameter_variables):
    assert gradient.shape == variable.shape
    assert np.all(np.isfinite(gradient.numpy()))


# Deterministic retraining

(
    repeated_embedding_matrix,
    repeated_update_input_weights,
    repeated_update_recurrent_weights,
    repeated_reset_input_weights,
    repeated_reset_recurrent_weights,
    repeated_candidate_input_weights,
    repeated_candidate_recurrent_weights,
    repeated_output_weights,
    repeated_train_history,
    repeated_validation_history
) = train_model(
    train_inputs,
    train_targets,
    validation_inputs,
    validation_targets,
    embedding_matrix,
    update_input_weights,
    update_recurrent_weights,
    reset_input_weights,
    reset_recurrent_weights,
    candidate_input_weights,
    candidate_recurrent_weights,
    output_weights,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    seed=SEED,
    print_every=None
)

repeated_parameter_variables = [
    repeated_embedding_matrix,
    repeated_update_input_weights,
    repeated_update_recurrent_weights,
    repeated_reset_input_weights,
    repeated_reset_recurrent_weights,
    repeated_candidate_input_weights,
    repeated_candidate_recurrent_weights,
    repeated_output_weights
]

for trained_variable, repeated_variable in zip(trained_parameter_variables, repeated_parameter_variables):
    assert np.allclose(trained_variable.numpy(), repeated_variable.numpy())

assert np.allclose(train_loss_history, repeated_train_history)
assert np.allclose(validation_loss_history, repeated_validation_history)


# Parameter-update checks

for initial_variable, trained_variable in zip(initial_parameter_variables, trained_parameter_variables):
    assert np.max(np.abs(initial_variable.numpy() - trained_variable.numpy())) > 0.0


# Generation reproducibility

first_generation = generate_text(
    starting_context="mode",
    number_of_characters=30,
    embedding_matrix=trained_embedding_matrix,
    update_input_weights=trained_update_input_weights,
    update_recurrent_weights=trained_update_recurrent_weights,
    reset_input_weights=trained_reset_input_weights,
    reset_recurrent_weights=trained_reset_recurrent_weights,
    candidate_input_weights=trained_candidate_input_weights,
    candidate_recurrent_weights=trained_candidate_recurrent_weights,
    output_weights=trained_output_weights,
    seed=10
)

second_generation = generate_text(
    starting_context="mode",
    number_of_characters=30,
    embedding_matrix=trained_embedding_matrix,
    update_input_weights=trained_update_input_weights,
    update_recurrent_weights=trained_update_recurrent_weights,
    reset_input_weights=trained_reset_input_weights,
    reset_recurrent_weights=trained_reset_recurrent_weights,
    candidate_input_weights=trained_candidate_input_weights,
    candidate_recurrent_weights=trained_candidate_recurrent_weights,
    output_weights=trained_output_weights,
    seed=10
)

assert first_generation == second_generation
assert len(first_generation) == CONTEXT_LENGTH + 30
assert first_generation.startswith("mode")

print("All checks passed.")

All checks passed.


## Notes

- The model remains a character-level neural language model.
- The training corpus and vocabulary remain unchanged from Version 6.
- Each prediction still uses a fixed context of four previous characters.
- Character identifiers are mapped to trainable 8-dimensional embedding vectors.
- The context embeddings are processed sequentially from left to right.
- Version 7 replaces the simple recurrent update from Version 6 with a gated recurrent unit.
- The GRU uses an update gate, a reset gate and a candidate hidden state.
- The update and reset gates use sigmoid activations.
- The candidate hidden state uses `tanh`.
- The hidden state starts from zeros for every training example.
- No bias terms are used.
- The model learns eight trainable parameter matrices:
  - the embedding matrix;
  - the update-gate input weights;
  - the update-gate recurrent weights;
  - the reset-gate input weights;
  - the reset-gate recurrent weights;
  - the candidate-state input weights;
  - the candidate-state recurrent weights;
  - the hidden-to-output weight matrix.
- The same GRU weights are reused at every position in the context.
- Gradients for all trainable parameters are computed automatically with `tf.GradientTape`, and the parameters are updated manually with gradient descent.
- The model contains 1800 trainable parameters, compared with 1032 in Version 6.
- Training and validation examples remain reproducibly separated.
- Training examples are shuffled before each sequence of mini-batch updates.
- Mini-batches are used to update the model parameters.
- Training loss decreases from approximately `3.2958` to `2.6596`.
- Validation loss decreases from approximately `3.2958` to `2.9139`.
- The best validation loss is approximately `2.8995` at epoch 99.
- Validation loss increases slightly at the final epoch after reaching its minimum, suggesting the beginning of mild overfitting.
- The final epoch-100 parameters are intentionally retained for generation.
- Generation remains autoregressive and uses a sliding four-character context window.
- Each generation window is processed by the GRU from a zero hidden state, matching the training procedure.
- The same seed makes parameter initialization, data splitting, mini-batch shuffling, training and generation reproducible within the same software environment.
- The tests verify GRU parameter shapes, gate values, hidden states, probability distributions, gradients, parameter updates and reproducibility.

Version 7 introduces gated recurrent processing while preserving the fixed four-character sequence-modeling setup introduced in the previous versions.

The GRU gives the model explicit learned mechanisms for controlling how previous information contributes to each hidden-state update.

The hidden state still carries information only through the characters inside each four-character context. It is not preserved across the complete training corpus or indefinitely during generation.

The fixed context, small corpus, absence of bias terms and lack of early stopping or checkpoint restoration intentionally keep the model easy to inspect.

Version 7 therefore completes the gated recurrent stage of the project and prepares the transition from recurrent sequence processing to attention mechanisms in the next version.

Future versions will introduce new components and gradually evolve the architecture.